In [1]:
import ast
import json
import os
import pandas as pd

In [2]:
counterfactual_analogy = os.path.join(os.path.dirname(os.getcwd()), "..", "counterfactual_analogy")

In [3]:
data_path = os.path.join(counterfactual_analogy + "/data/data.csv")
data = pd.read_csv(data_path)
# remove symbol alphabet:
data = data[data["nperms"] != "np_symb"]
# remove attention check tasks:
data = data[data["prob_type"] != "attn"]
data = data.rename(columns={"prob_type": "transformation", "nperms": "n_perm"})

In [4]:
problem_files = [file for file in os.listdir(counterfactual_analogy + "/problems/zerogen") if ".json" in file ]
problem_files

['all_prob_10_7.json',
 'all_prob_1_7.json',
 'all_prob_20_7.json',
 'all_prob_2_7.json',
 'all_prob_5_7.json',
 'all_prob_symb_7.json']

In [5]:
nps = [1,2,5,10,20]

mappings = {}
for np in nps:
    file_path = os.path.join(counterfactual_analogy, f"problems/zerogen/all_prob_{np}_7.json")
    mappings[f"np_{np}"] = {}
    with open(file_path, "r") as f:
        js_file = json.load(f)
        for key, value in js_file.items():
            if key.startswith("alph_") and "shuffled_alphabet" in value:
                mappings[f"np_{np}"][key] = value["shuffled_alphabet"]

In [7]:
data["alphabet"] = data.apply(lambda row: mappings[row["n_perm"]][row["alph"]], axis=1)

In [8]:
# convert all relevant columns to lists
data["source_1"] = data["source_1"].apply(ast.literal_eval)
data["source_2"] = data["source_2"].apply(ast.literal_eval)
data["target_1"] = data["target_1"].apply(ast.literal_eval)
data["correct_answer"] = data["correct_answer"].apply(ast.literal_eval)

# remove prefix in n_perm:
data["n_perm"] = data["n_perm"].apply(lambda x: int(x.replace("np_", "")))
# add missing columns:
data["query_length"] = data["target_1"].apply(len)
data["copy"] = False
data["distribution"] = "in"

In [9]:
# convert to datasets layout
data["alphabet"] = data["alphabet"].apply(lambda x: " ".join(x))
data["study"] = data.apply(lambda row: " ".join(row["source_1"]) + " > " + " ".join(row["source_2"]), axis=1)
data["problem"] = data.apply(lambda row: " ".join(row["target_1"]) + " > " + " ".join(row["correct_answer"]), axis=1)

In [10]:
data.drop(
    columns=[
        'subj_id',
        'model',
        'promptstyle',
        'alph',
        'prob_ind',
        'response_string',
        'source_1',
        'source_2',
        'target_1',
        'correct_answer',
        'given_answer',
        'correct',
        'total'], 
    inplace=True
)
data = data[["n_perm", "alphabet", "transformation", "problem", "query_length", "study", "distribution", "copy"]]
data

,n_perm,alphabet,transformation,problem,query_length,study,distribution,copy
0,10,a b c q e f g y i o k l t n x p h r s u d v w ...,succ,g y i o > g y i k,4,x p h r > x p h s,in,False
1,10,e a c d b f g r i j k w m p n h q l s t u v o ...,add_letter,f g r i > f g r i j,4,q l s t > q l s t u,in,False
3,10,a q c d e o g h i n k l m u f p y r s t w v b ...,fix_alphabet,e o g a i > e o g h i,5,k r m u f > k l m u f,in,False
4,10,c b v h e f g d r j k l m n o p q y s t u x a ...,remove_redundant,v h h e f g > v h e f g,6,x x a w i z > x a w i z,in,False
5,10,a b s d e f g o i j k c x n h p q t y r u v w ...,sort,p y t q r > p q t y r,5,m w l v z > v w l m z,in,False
...,...,...,...,...,...,...,...,...
26009,1,a b c d e f g h i j k l m n o p q r s t u v w ...,sort,h l j k i > h i j k l,5,u v y x w > u v w x y,in,False
26010,1,a b c d e f g h i j k l m n o p q r s t u v w ...,sort,q t s r u > q r s t u,5,r p q o s > o p q r s,in,False
26011,1,a b c d e f g h i j k l m n o p q r s t u v w ...,sort,k h i j g > g h i j k,5,h k j i l > h i j k l,in,False
26012,1,a b c d e f g h i j k l m n o p q r s t u v w ...,sort,s p q r o > o p q r s,5,r s u t v > r s t u v,in,False


In [11]:
# only keep unique tasks:
data["identifier"] = data.apply(lambda row: row["alphabet"] + row["study"] + row["problem"], axis=1)
data

,n_perm,alphabet,transformation,problem,query_length,study,distribution,copy,identifier
0,10,a b c q e f g y i o k l t n x p h r s u d v w ...,succ,g y i o > g y i k,4,x p h r > x p h s,in,False,a b c q e f g y i o k l t n x p h r s u d v w ...
1,10,e a c d b f g r i j k w m p n h q l s t u v o ...,add_letter,f g r i > f g r i j,4,q l s t > q l s t u,in,False,e a c d b f g r i j k w m p n h q l s t u v o ...
3,10,a q c d e o g h i n k l m u f p y r s t w v b ...,fix_alphabet,e o g a i > e o g h i,5,k r m u f > k l m u f,in,False,a q c d e o g h i n k l m u f p y r s t w v b ...
4,10,c b v h e f g d r j k l m n o p q y s t u x a ...,remove_redundant,v h h e f g > v h e f g,6,x x a w i z > x a w i z,in,False,c b v h e f g d r j k l m n o p q y s t u x a ...
5,10,a b s d e f g o i j k c x n h p q t y r u v w ...,sort,p y t q r > p q t y r,5,m w l v z > v w l m z,in,False,a b s d e f g o i j k c x n h p q t y r u v w ...
...,...,...,...,...,...,...,...,...,...
26009,1,a b c d e f g h i j k l m n o p q r s t u v w ...,sort,h l j k i > h i j k l,5,u v y x w > u v w x y,in,False,a b c d e f g h i j k l m n o p q r s t u v w ...
26010,1,a b c d e f g h i j k l m n o p q r s t u v w ...,sort,q t s r u > q r s t u,5,r p q o s > o p q r s,in,False,a b c d e f g h i j k l m n o p q r s t u v w ...
26011,1,a b c d e f g h i j k l m n o p q r s t u v w ...,sort,k h i j g > g h i j k,5,h k j i l > h i j k l,in,False,a b c d e f g h i j k l m n o p q r s t u v w ...
26012,1,a b c d e f g h i j k l m n o p q r s t u v w ...,sort,s p q r o > o p q r s,5,r s u t v > r s t u v,in,False,a b c d e f g h i j k l m n o p q r s t u v w ...


In [12]:
data = data.drop_duplicates(subset=["identifier"])
data

,n_perm,alphabet,transformation,problem,query_length,study,distribution,copy,identifier
0,10,a b c q e f g y i o k l t n x p h r s u d v w ...,succ,g y i o > g y i k,4,x p h r > x p h s,in,False,a b c q e f g y i o k l t n x p h r s u d v w ...
1,10,e a c d b f g r i j k w m p n h q l s t u v o ...,add_letter,f g r i > f g r i j,4,q l s t > q l s t u,in,False,e a c d b f g r i j k w m p n h q l s t u v o ...
3,10,a q c d e o g h i n k l m u f p y r s t w v b ...,fix_alphabet,e o g a i > e o g h i,5,k r m u f > k l m u f,in,False,a q c d e o g h i n k l m u f p y r s t w v b ...
4,10,c b v h e f g d r j k l m n o p q y s t u x a ...,remove_redundant,v h h e f g > v h e f g,6,x x a w i z > x a w i z,in,False,c b v h e f g d r j k l m n o p q y s t u x a ...
5,10,a b s d e f g o i j k c x n h p q t y r u v w ...,sort,p y t q r > p q t y r,5,m w l v z > v w l m z,in,False,a b s d e f g o i j k c x n h p q t y r u v w ...
...,...,...,...,...,...,...,...,...,...
4573,20,x p j h n a u d i q k c m t f s b r g y e v w ...,fix_alphabet,s b r k y > s b r g y,5,i q r c m > i q k c m,in,False,x p j h n a u d i q k c m t f s b r g y e v w ...
4577,20,x p j h n a u d i q k c m t f s b r g y e v w ...,sort,n h a u d > h n a u d,5,a h n j u > j h n a u,in,False,x p j h n a u d i q k c m t f s b r g y e v w ...
4579,20,x p j h n a u d i q k c m t f s b r g y e v w ...,sort,k t m c f > k c m t f,5,i d u q k > u d i q k,in,False,x p j h n a u d i q k c m t f s b r g y e v w ...
4581,20,x p j h n a u d i q k c m t f s b r g y e v w ...,sort,t k c m q > q k c m t,5,l w z v o > v w z l o,in,False,x p j h n a u d i q k c m t f s b r g y e v w ...


In [13]:
# drop identifier and write data to folder:
data.drop(columns=["identifier"], inplace=True)

# write to csv
os.mkdir("lewis_mitchell")
data.to_csv("lewis_mitchell/test.csv", index=False)

C:\Users\13579681\AppData\Local\Temp\ipykernel_36456\4130078216.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.drop(columns=["identifier"], inplace=True)
